# Initialisation

In [177]:
!pip install kaggle

In [178]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("gauravmalik26/food-delivery-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'food-delivery-dataset' dataset.
Path to dataset files: /kaggle/input/food-delivery-dataset


In [179]:
import pandas as pd
import os

# 'path' contains the directory where the dataset files are extracted.
# We need to find the 'train.csv' file within this directory and load it.
file_name = "train.csv"
full_path = os.path.join(path, file_name)

# Check if the file exists before attempting to read it
if os.path.exists(full_path):
    df = pd.read_csv(full_path)
    print("DataFrame 'df' created successfully.")
    print(df.shape)
else:
    print(f"Erreur: Le fichier '{file_name}' est introuvable à l'emplacement: {full_path}")

DataFrame 'df' created successfully.
(45593, 20)



# EDA

In [180]:
df.info()
df.describe()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ID                           45593 non-null  object 
 1   Delivery_person_ID           45593 non-null  object 
 2   Delivery_person_Age          45593 non-null  object 
 3   Delivery_person_Ratings      45593 non-null  object 
 4   Restaurant_latitude          45593 non-null  float64
 5   Restaurant_longitude         45593 non-null  float64
 6   Delivery_location_latitude   45593 non-null  float64
 7   Delivery_location_longitude  45593 non-null  float64
 8   Order_Date                   45593 non-null  object 
 9   Time_Orderd                  45593 non-null  object 
 10  Time_Order_picked            45593 non-null  object 
 11  Weatherconditions            45593 non-null  object 
 12  Road_traffic_density         45593 non-null  object 
 13  Vehicle_conditio

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


In [181]:
import numpy as np

df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

df = df.replace({
    '': np.nan,
    ' ': np.nan,
    'NaN': np.nan,
    'nan': np.nan,
    'NULL': np.nan,
    'null': np.nan,
    'None': np.nan
})

/tmp/ipykernel_7695/92443957.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [182]:
df.isna().sum()

,0
ID,0
Delivery_person_ID,0
Delivery_person_Age,1854
Delivery_person_Ratings,1908
Restaurant_latitude,0
Restaurant_longitude,0
Delivery_location_latitude,0
Delivery_location_longitude,0
Order_Date,0
Time_Orderd,1731


In [183]:
df.duplicated().sum()
df[df.duplicated()]

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)


# Nettoyage

In [184]:
# -----------------------------
# 3. NETTOYAGE DES FAUX NAN
# -----------------------------
# Normalize column names to lowercase to avoid case-sensitivity issues
df.columns = df.columns.str.strip().str.lower()
df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

df = df.replace({
    '': np.nan,
    ' ': np.nan,
    'nan': np.nan,
    'NaN': np.nan,
    'NULL': np.nan,
    'null': np.nan,
    'None': np.nan
})

# regex pour espaces invisibles
df = df.replace(r'^\s*$', np.nan, regex=True)

# -----------------------------
# 4. SUPPRESSION DES DOUBLONS
# -----------------------------
df = df.drop_duplicates()

# -----------------------------
# 5. CONVERSION DES TYPES (Moved here, before imputation)
# -----------------------------

# numériques
cols_num = [
    'delivery_person_age',
    'delivery_person_ratings',
    'restaurant_latitude',
    'restaurant_longitude',
    'delivery_location_latitude',
    'delivery_location_longitude',
    'vehicle_condition',
    'multiple_deliveries'
]

for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Missing values before imputation:")
print(df.isna().sum())

# Impute numerical columns with the median
for col in ['delivery_person_age', 'delivery_person_ratings', 'multiple_deliveries']:
    if col in df.columns:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled missing values in '{col}' with median: {median_val}")

# Impute categorical/object columns with the mode
for col in ['time_orderd', 'road_traffic_density', 'festival', 'city']:
    if col in df.columns:
        mode_val = df[col].mode()[0] # .mode() returns a Series, take the first if multiple modes
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled missing values in '{col}' with mode: {mode_val}")

print("\nMissing values after imputation:")
print(df.isna().sum())

/tmp/ipykernel_7695/3673106179.py:6: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


Missing values before imputation:
id                                0
delivery_person_id                0
delivery_person_age            1854
delivery_person_ratings        1908
restaurant_latitude               0
restaurant_longitude              0
delivery_location_latitude        0
delivery_location_longitude       0
order_date                        0
time_orderd                    1731
time_order_picked                 0
weatherconditions                 0
road_traffic_density            601
vehicle_condition                 0
type_of_order                     0
type_of_vehicle                   0
multiple_deliveries             993
festival                        228
city                           1200
time_taken(min)                   0
dtype: int64
Filled missing values in 'delivery_person_age' with median: 30.0
Filled missing values in 'delivery_person_ratings' with median: 4.7
Filled missing values in 'multiple_deliveries' with median: 1.0
Filled missing values in 'time_order

/tmp/ipykernel_7695/3673106179.py:52: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
/tmp/ipykernel_7695/3673106179.py:59: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

# Feature engineering

In [185]:
df['order_datetime'] = pd.to_datetime(df['order_date'] + ' ' + df['time_orderd'])

/tmp/ipykernel_7695/4169523137.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['order_datetime'] = pd.to_datetime(df['order_date'] + ' ' + df['time_orderd'])


In [186]:
df['order_datetime'] = pd.to_datetime(df['order_datetime'])

In [187]:
df['time_taken(min)'] = df['time_taken(min)'].str.replace('(min)', '', regex=False).astype(int)
df['is_late'] = df['time_taken(min)'] > 30

In [188]:
df['hour'] = df['order_datetime'].dt.hour

In [189]:
orders_per_courier = df.groupby('delivery_person_id').size()

In [190]:
facteurs = {
    'bike': 0,
    'electric_scooter': 20,
    'scooter': 70,
    'motorcycle': 90
}

df['emission_factor'] = df['type_of_vehicle'].map(facteurs).fillna(70)

In [191]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # rayon Terre en km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [192]:
df['distance_km'] = haversine(
    df['restaurant_latitude'],
    df['restaurant_longitude'],
    df['delivery_location_latitude'],
    df['delivery_location_longitude']
)

In [193]:
df = df.dropna(subset=[
    'restaurant_latitude',
    'restaurant_longitude',
    'delivery_location_latitude',
    'delivery_location_longitude'
])

In [194]:
df['distance_km'] = df['distance_km'] * 1.3

In [195]:
df['co2_emission'] = df['distance_km'] * df['emission_factor']

In [196]:
df['co2_emission'] = df['distance_km'] * df['emission_factor']

# Export

In [197]:
df.to_csv('delivery_data.csv', index=False)